
# Grain Robustness Q1 — Notebook 04
## Selección de modelos restringida al conjunto de entrenamiento

Este notebook utiliza exclusivamente:

`NOTEBOOK03_FROZEN_SPLITS_OUTPUTS.zip`

### Objetivo

Seleccionar una configuración moderada para cuatro familias de modelos,
tratando el algoritmo como **factor de sensibilidad** y la representación
morfométrica como **factor científico principal**.

### Familias evaluadas

1. Regresión logística multinomial.
2. SVM con kernel radial.
3. Extra Trees.
4. CatBoost.

### Reglas metodológicas

- Solo se utiliza la partición `train`.
- Los folds internos ya congelados se usan como validación cruzada.
- `calibration` y `test` no participan en ningún cálculo de selección.
- Los grupos duplicados permanecen intactos.
- La métrica primaria es macro-F1.
- Los grids son moderados y predefinidos.
- Se selecciona una configuración por dataset, representación y algoritmo.
- No se informa todavía desempeño final en prueba.
- Utilizar **CPU**. La GPU no es necesaria.

### Tiempo estimado en Colab CPU

Entre 20 y 90 minutos, dependiendo del tipo de CPU asignado por Colab.
El SVM suele ser la etapa más lenta.


## 1. Preparar Colab

In [ ]:

!pip -q install pyarrow catboost joblib


In [ ]:

from google.colab import drive
drive.mount('/content/drive', force_remount=True)


## 2. Importaciones y configuración

In [ ]:

from pathlib import Path
from datetime import datetime, timezone
from itertools import product
import hashlib
import json
import math
import os
import platform
import shutil
import sys
import time
import traceback
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    log_loss,
)
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')

RANDOM_SEED = 20260803

LOCAL_ROOT = Path('/content/Grain_Robustness_Q1_notebook04')
INPUT_DIR = LOCAL_ROOT / '00_input'
EXTRACT_DIR = LOCAL_ROOT / '01_extracted_notebook03'
OUTPUT_DIR = LOCAL_ROOT / '02_outputs'
RESULTS_DIR = OUTPUT_DIR / '07_results' / '04_MODEL_SELECTION'
TABLES_DIR = OUTPUT_DIR / '09_tables' / '04_MODEL_SELECTION'
FIGURES_DIR = OUTPUT_DIR / '08_figures' / '04_MODEL_SELECTION'
PROTOCOL_DIR = OUTPUT_DIR / '03_protocol'

DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/Grain_Robustness_Q1')
DRIVE_CHECKPOINT_DIR = (
    DRIVE_PROJECT_ROOT / '07_results' / '04_MODEL_SELECTION_CHECKPOINTS'
)

for directory in [
    INPUT_DIR,
    EXTRACT_DIR,
    RESULTS_DIR,
    TABLES_DIR,
    FIGURES_DIR,
    PROTOCOL_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

DATASETS = ['dry_bean', 'iniap_rice']
REPRESENTATIONS = [
    'R1_full_native',
    'R2_measured_core',
    'R3_invariant_core',
    'R4_hybrid_log_area',
]
ALGORITHMS = [
    'multinomial_logistic_regression',
    'rbf_svm',
    'extra_trees',
    'catboost',
]

METADATA_COLUMNS = {
    'dataset_id',
    'record_id',
    'source_excel_row',
    'feature_fingerprint_sha256',
    'full_row_fingerprint_sha256',
    'duplicate_group_id',
    'exact_duplicate_count',
    'class_label',
    'fold_id',
    'partition',
}

print('Espacio local:', LOCAL_ROOT)
print('CPU disponibles:', os.cpu_count())
print('Python:', sys.version.split()[0])


## 3. Localizar el ZIP del Notebook 03

In [ ]:

from google.colab import files

candidate_paths = [
    Path('/content/NOTEBOOK03_FROZEN_SPLITS_OUTPUTS.zip'),
    (
        DRIVE_PROJECT_ROOT / '07_results'
        / 'NOTEBOOK03_FROZEN_SPLITS_OUTPUTS.zip'
    ),
    (
        DRIVE_PROJECT_ROOT
        / 'NOTEBOOK03_FROZEN_SPLITS_OUTPUTS.zip'
    ),
]

zip_source = next(
    (path for path in candidate_paths if path.exists()),
    None,
)

if zip_source is None:
    print(
        'Selecciona NOTEBOOK03_FROZEN_SPLITS_OUTPUTS.zip.'
    )
    uploaded = files.upload()
    uploaded_names = list(uploaded.keys())

    if len(uploaded_names) != 1:
        raise RuntimeError(
            'Debes cargar exactamente un archivo ZIP.'
        )

    zip_source = Path('/content') / uploaded_names[0]

if zip_source.suffix.lower() != '.zip':
    raise ValueError(f'El archivo no es ZIP: {zip_source}')

zip_local = INPUT_DIR / 'NOTEBOOK03_FROZEN_SPLITS_OUTPUTS.zip'
shutil.copy2(zip_source, zip_local)

print('ZIP utilizado:', zip_source)
print('Copia local:', zip_local)
print('Tamaño MB:', round(zip_local.stat().st_size / 1_000_000, 3))


## 4. Extraer y validar la entrada

In [ ]:

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_local, 'r') as archive:
    archive.extractall(EXTRACT_DIR)

validation_path = (
    EXTRACT_DIR / '07_results' / '03_FROZEN_SPLITS'
    / '06_final_split_validation.csv'
)
leakage_path = (
    EXTRACT_DIR / '07_results' / '03_FROZEN_SPLITS'
    / '02_split_leakage_audit.csv'
)
protocol_path = (
    EXTRACT_DIR / '03_protocol'
    / 'frozen_split_and_modeling_protocol_v1.json'
)

required = [validation_path, leakage_path, protocol_path]

for dataset_id in DATASETS:
    for representation in REPRESENTATIONS:
        required.append(
            EXTRACT_DIR / '05_frozen_splits'
            / dataset_id
            / f'{representation}_with_split.parquet'
        )

missing = [str(path) for path in required if not path.exists()]

if missing:
    raise FileNotFoundError(
        'Faltan archivos requeridos:\n' + '\n'.join(missing)
    )

previous_validation = pd.read_csv(validation_path)
leakage_audit = pd.read_csv(leakage_path)

if not previous_validation['passed'].all():
    raise RuntimeError(
        'La salida del Notebook 03 no superó todas las validaciones.'
    )

numeric_leakage_columns = [
    column for column in leakage_audit.columns
    if column != 'dataset_id'
]

if leakage_audit[numeric_leakage_columns].to_numpy().sum() != 0:
    raise RuntimeError(
        'La entrada contiene indicios de fuga entre particiones.'
    )

print('Salida del Notebook 03 validada correctamente.')
display(previous_validation)
display(leakage_audit)


## 5. Cargar las representaciones y aislar entrenamiento

In [ ]:

datasets = {}
feature_columns = {}
train_fold_ids = {}

for dataset_id in DATASETS:
    datasets[dataset_id] = {}
    feature_columns[dataset_id] = {}

    reference_keys = None

    for representation in REPRESENTATIONS:
        path = (
            EXTRACT_DIR / '05_frozen_splits'
            / dataset_id
            / f'{representation}_with_split.parquet'
        )
        dataframe = pd.read_parquet(path)
        dataframe = dataframe.sort_values('record_id').reset_index(drop=True)

        keys = dataframe[
            [
                'record_id',
                'class_label',
                'duplicate_group_id',
                'fold_id',
                'partition',
            ]
        ].copy()

        if reference_keys is None:
            reference_keys = keys
        elif not reference_keys.equals(keys):
            raise RuntimeError(
                f'Las representaciones no están alineadas: '
                f'{dataset_id}, {representation}'
            )

        features = [
            column for column in dataframe.columns
            if column not in METADATA_COLUMNS
        ]

        if not features:
            raise RuntimeError(
                f'No se detectaron predictores en {dataset_id}, '
                f'{representation}.'
            )

        if dataframe[features].isna().sum().sum() != 0:
            raise RuntimeError(
                f'Hay valores faltantes en {dataset_id}, '
                f'{representation}.'
            )

        datasets[dataset_id][representation] = dataframe
        feature_columns[dataset_id][representation] = features

    train_folds = sorted(
        reference_keys.loc[
            reference_keys['partition'] == 'train',
            'fold_id',
        ].unique().tolist()
    )

    if len(train_folds) != 3:
        raise RuntimeError(
            f'Se esperaban tres folds internos de entrenamiento en '
            f'{dataset_id}; se encontraron {train_folds}.'
        )

    train_fold_ids[dataset_id] = train_folds

    print(
        dataset_id,
        'folds internos de entrenamiento:',
        train_folds,
    )


## 6. Grids moderados y predefinidos

In [ ]:

PARAMETER_GRIDS = {
    'multinomial_logistic_regression': [
        {'C': 0.1},
        {'C': 1.0},
        {'C': 10.0},
    ],
    'rbf_svm': [
        {'C': 1.0, 'gamma': 'scale'},
        {'C': 10.0, 'gamma': 'scale'},
        {'C': 1.0, 'gamma': 0.1},
        {'C': 10.0, 'gamma': 0.1},
    ],
    'extra_trees': [
        {'max_features': 'sqrt', 'min_samples_leaf': 1},
        {'max_features': 'sqrt', 'min_samples_leaf': 3},
        {'max_features': 1.0, 'min_samples_leaf': 1},
        {'max_features': 1.0, 'min_samples_leaf': 3},
    ],
    'catboost': [
        {'depth': 4, 'learning_rate': 0.03},
        {'depth': 4, 'learning_rate': 0.10},
        {'depth': 6, 'learning_rate': 0.03},
        {'depth': 6, 'learning_rate': 0.10},
    ],
}

grid_catalog_rows = []

for algorithm, candidates in PARAMETER_GRIDS.items():
    for candidate_index, parameters in enumerate(candidates):
        grid_catalog_rows.append({
            'algorithm': algorithm,
            'candidate_id': f'{algorithm}_C{candidate_index + 1:02d}',
            'parameters_json': json.dumps(
                parameters,
                sort_keys=True,
            ),
            'complexity_order': candidate_index + 1,
        })

grid_catalog = pd.DataFrame(grid_catalog_rows)
grid_catalog.to_csv(
    PROTOCOL_DIR / 'model_selection_grid_catalog.csv',
    index=False,
    encoding='utf-8-sig',
)

display(grid_catalog)


## 7. Funciones de construcción y evaluación

In [ ]:

def build_estimator(algorithm, parameters, random_seed):
    if algorithm == 'multinomial_logistic_regression':
        model = LogisticRegression(
            C=parameters['C'],
            solver='lbfgs',
            max_iter=5000,
            random_state=random_seed,
        )
        return Pipeline([
            ('scaler', StandardScaler()),
            ('model', model),
        ])

    if algorithm == 'rbf_svm':
        model = SVC(
            C=parameters['C'],
            gamma=parameters['gamma'],
            kernel='rbf',
            probability=True,
            random_state=random_seed,
            cache_size=2000,
        )
        return Pipeline([
            ('scaler', StandardScaler()),
            ('model', model),
        ])

    if algorithm == 'extra_trees':
        return ExtraTreesClassifier(
            n_estimators=300,
            max_features=parameters['max_features'],
            min_samples_leaf=parameters['min_samples_leaf'],
            random_state=random_seed,
            n_jobs=-1,
        )

    if algorithm == 'catboost':
        return CatBoostClassifier(
            iterations=400,
            depth=parameters['depth'],
            learning_rate=parameters['learning_rate'],
            loss_function='MultiClass',
            random_seed=random_seed,
            verbose=False,
            allow_writing_files=False,
            thread_count=-1,
        )

    raise ValueError(f'Algoritmo no reconocido: {algorithm}')


def aligned_probabilities(estimator, x, global_classes):
    probabilities = estimator.predict_proba(x)
    estimator_classes = np.asarray(estimator.classes_)

    aligned = np.zeros(
        (len(x), len(global_classes)),
        dtype=float,
    )

    class_to_position = {
        class_label: position
        for position, class_label in enumerate(global_classes)
    }

    for source_position, class_label in enumerate(estimator_classes):
        aligned[:, class_to_position[class_label]] = (
            probabilities[:, source_position]
        )

    return aligned


def evaluate_candidate(
    dataframe,
    features,
    algorithm,
    parameters,
    internal_folds,
):
    train_only = dataframe[
        dataframe['partition'] == 'train'
    ].copy()

    global_classes = np.sort(
        train_only['class_label'].astype(str).unique()
    )

    fold_rows = []
    start_time = time.perf_counter()

    for validation_fold in internal_folds:
        fit_mask = (
            train_only['fold_id'] != validation_fold
        )
        validation_mask = (
            train_only['fold_id'] == validation_fold
        )

        fit_data = train_only.loc[fit_mask]
        validation_data = train_only.loc[validation_mask]

        x_fit = fit_data[features].to_numpy(dtype=float)
        y_fit = fit_data['class_label'].astype(str).to_numpy()

        x_validation = validation_data[
            features
        ].to_numpy(dtype=float)
        y_validation = validation_data[
            'class_label'
        ].astype(str).to_numpy()

        estimator = build_estimator(
            algorithm,
            parameters,
            RANDOM_SEED,
        )
        estimator.fit(x_fit, y_fit)

        prediction = estimator.predict(x_validation)
        probabilities = aligned_probabilities(
            estimator,
            x_validation,
            global_classes,
        )

        fold_rows.append({
            'validation_fold': int(validation_fold),
            'n_fit': len(fit_data),
            'n_validation': len(validation_data),
            'accuracy': accuracy_score(
                y_validation,
                prediction,
            ),
            'balanced_accuracy': balanced_accuracy_score(
                y_validation,
                prediction,
            ),
            'macro_f1': f1_score(
                y_validation,
                prediction,
                average='macro',
            ),
            'mcc': matthews_corrcoef(
                y_validation,
                prediction,
            ),
            'log_loss': log_loss(
                y_validation,
                probabilities,
                labels=global_classes,
            ),
        })

    elapsed_seconds = time.perf_counter() - start_time

    return pd.DataFrame(fold_rows), elapsed_seconds


## 8. Ejecutar selección con puntos de control

In [ ]:

checkpoint_local = (
    RESULTS_DIR / 'model_selection_fold_results_checkpoint.csv'
)
checkpoint_drive = (
    DRIVE_CHECKPOINT_DIR
    / 'model_selection_fold_results_checkpoint.csv'
)


def copy_with_retry(source, destination, attempts=5):
    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            destination.parent.mkdir(
                parents=True,
                exist_ok=True,
            )
            shutil.copy2(source, destination)

            if (
                not destination.exists()
                or destination.stat().st_size
                != source.stat().st_size
            ):
                raise IOError(
                    'La copia no pudo verificarse.'
                )

            return True

        except Exception as error:
            last_error = error
            print(
                f'Checkpoint Drive {attempt}/{attempts}: '
                f'{type(error).__name__}: {error}'
            )
            time.sleep(3)

    print(
        'Advertencia: no se pudo actualizar el checkpoint en Drive. '
        f'Último error: {last_error}'
    )
    return False


if checkpoint_local.exists():
    completed_results = pd.read_csv(checkpoint_local)
else:
    completed_results = pd.DataFrame()

completed_keys = set()

if not completed_results.empty:
    completed_keys = set(
        zip(
            completed_results['dataset_id'],
            completed_results['representation'],
            completed_results['algorithm'],
            completed_results['candidate_id'],
        )
    )

all_rows = (
    completed_results.to_dict('records')
    if not completed_results.empty
    else []
)

total_candidates = (
    len(DATASETS)
    * len(REPRESENTATIONS)
    * sum(
        len(candidates)
        for candidates in PARAMETER_GRIDS.values()
    )
)
candidate_counter = len(completed_keys)

for dataset_id in DATASETS:
    for representation in REPRESENTATIONS:
        dataframe = datasets[dataset_id][representation]
        features = feature_columns[dataset_id][representation]

        print('\n' + '=' * 90)
        print(dataset_id, representation, 'features:', len(features))

        for algorithm in ALGORITHMS:
            candidates = PARAMETER_GRIDS[algorithm]

            for candidate_index, parameters in enumerate(candidates):
                candidate_id = (
                    f'{algorithm}_C{candidate_index + 1:02d}'
                )
                key = (
                    dataset_id,
                    representation,
                    algorithm,
                    candidate_id,
                )

                if key in completed_keys:
                    print('Omitido por checkpoint:', key)
                    continue

                candidate_counter += 1
                print(
                    f'[{candidate_counter}/{total_candidates}]',
                    dataset_id,
                    representation,
                    algorithm,
                    candidate_id,
                    parameters,
                )

                try:
                    fold_results, elapsed_seconds = evaluate_candidate(
                        dataframe=dataframe,
                        features=features,
                        algorithm=algorithm,
                        parameters=parameters,
                        internal_folds=train_fold_ids[dataset_id],
                    )

                    for _, fold_row in fold_results.iterrows():
                        all_rows.append({
                            'dataset_id': dataset_id,
                            'representation': representation,
                            'algorithm': algorithm,
                            'candidate_id': candidate_id,
                            'parameters_json': json.dumps(
                                parameters,
                                sort_keys=True,
                            ),
                            'feature_count': len(features),
                            'validation_fold': int(
                                fold_row['validation_fold']
                            ),
                            'n_fit': int(fold_row['n_fit']),
                            'n_validation': int(
                                fold_row['n_validation']
                            ),
                            'accuracy': float(
                                fold_row['accuracy']
                            ),
                            'balanced_accuracy': float(
                                fold_row['balanced_accuracy']
                            ),
                            'macro_f1': float(
                                fold_row['macro_f1']
                            ),
                            'mcc': float(fold_row['mcc']),
                            'log_loss': float(
                                fold_row['log_loss']
                            ),
                            'candidate_total_seconds': float(
                                elapsed_seconds
                            ),
                            'status': 'completed',
                        })

                except Exception as error:
                    print(
                        'ERROR:',
                        type(error).__name__,
                        error,
                    )
                    traceback.print_exc()

                    all_rows.append({
                        'dataset_id': dataset_id,
                        'representation': representation,
                        'algorithm': algorithm,
                        'candidate_id': candidate_id,
                        'parameters_json': json.dumps(
                            parameters,
                            sort_keys=True,
                        ),
                        'feature_count': len(features),
                        'validation_fold': np.nan,
                        'n_fit': np.nan,
                        'n_validation': np.nan,
                        'accuracy': np.nan,
                        'balanced_accuracy': np.nan,
                        'macro_f1': np.nan,
                        'mcc': np.nan,
                        'log_loss': np.nan,
                        'candidate_total_seconds': np.nan,
                        'status': (
                            f'failed: {type(error).__name__}: {error}'
                        ),
                    })

                checkpoint_dataframe = pd.DataFrame(all_rows)
                checkpoint_dataframe.to_csv(
                    checkpoint_local,
                    index=False,
                    encoding='utf-8-sig',
                )

        # Sincronización después de cada dataset-representación.
        copy_with_retry(
            checkpoint_local,
            checkpoint_drive,
        )

fold_results_all = pd.DataFrame(all_rows)

fold_results_all.to_csv(
    RESULTS_DIR / '01_model_selection_fold_results.csv',
    index=False,
    encoding='utf-8-sig',
)

print('Evaluación de candidatos terminada.')
print('Filas guardadas:', len(fold_results_all))


## 9. Resumir candidatos y seleccionar configuraciones

In [ ]:

completed = fold_results_all[
    fold_results_all['status'] == 'completed'
].copy()

candidate_summary = (
    completed.groupby(
        [
            'dataset_id',
            'representation',
            'algorithm',
            'candidate_id',
            'parameters_json',
            'feature_count',
        ]
    )
    .agg(
        folds=('validation_fold', 'nunique'),
        macro_f1_mean=('macro_f1', 'mean'),
        macro_f1_sd=('macro_f1', 'std'),
        macro_f1_min=('macro_f1', 'min'),
        balanced_accuracy_mean=(
            'balanced_accuracy',
            'mean',
        ),
        mcc_mean=('mcc', 'mean'),
        log_loss_mean=('log_loss', 'mean'),
        accuracy_mean=('accuracy', 'mean'),
        total_seconds=('candidate_total_seconds', 'first'),
    )
    .reset_index()
)

if not (candidate_summary['folds'] == 3).all():
    incomplete = candidate_summary[
        candidate_summary['folds'] != 3
    ]
    raise RuntimeError(
        'Hay candidatos con menos de tres folds completos:\n'
        + incomplete.to_string(index=False)
    )

candidate_summary.to_csv(
    RESULTS_DIR / '02_model_selection_candidate_summary.csv',
    index=False,
    encoding='utf-8-sig',
)

selected_parts = []

for (
    dataset_id,
    representation,
    algorithm,
), part in candidate_summary.groupby(
    ['dataset_id', 'representation', 'algorithm'],
    sort=True,
):
    selected = (
        part.sort_values(
            [
                'macro_f1_mean',
                'macro_f1_min',
                'log_loss_mean',
                'macro_f1_sd',
                'candidate_id',
            ],
            ascending=[
                False,
                False,
                True,
                True,
                True,
            ],
        )
        .head(1)
        .copy()
    )

    selected_parts.append(selected)

selected_configurations = pd.concat(
    selected_parts,
    ignore_index=True,
)

selected_configurations.to_csv(
    RESULTS_DIR / '03_selected_model_configurations.csv',
    index=False,
    encoding='utf-8-sig',
)

selected_json = {}

for _, row in selected_configurations.iterrows():
    dataset_id = row['dataset_id']
    representation = row['representation']
    algorithm = row['algorithm']

    selected_json.setdefault(dataset_id, {})
    selected_json[dataset_id].setdefault(
        representation,
        {},
    )
    selected_json[dataset_id][representation][algorithm] = {
        'candidate_id': row['candidate_id'],
        'parameters': json.loads(
            row['parameters_json']
        ),
        'cv_macro_f1_mean': float(
            row['macro_f1_mean']
        ),
        'cv_macro_f1_sd': float(
            row['macro_f1_sd']
        ),
        'cv_macro_f1_min': float(
            row['macro_f1_min']
        ),
        'cv_log_loss_mean': float(
            row['log_loss_mean']
        ),
    }

selected_json_path = (
    PROTOCOL_DIR
    / 'selected_model_configurations_v1.json'
)
selected_json_path.write_text(
    json.dumps(
        selected_json,
        indent=2,
        ensure_ascii=False,
    ),
    encoding='utf-8',
)

display(
    selected_configurations.sort_values(
        ['dataset_id', 'representation', 'algorithm']
    )
)


## 10. Resumen por representación sin declarar un algoritmo ganador

In [ ]:

representation_summary = (
    selected_configurations.groupby(
        ['dataset_id', 'representation']
    )
    .agg(
        algorithms=('algorithm', 'nunique'),
        macro_f1_across_algorithms_mean=(
            'macro_f1_mean',
            'mean',
        ),
        macro_f1_across_algorithms_sd=(
            'macro_f1_mean',
            'std',
        ),
        macro_f1_across_algorithms_min=(
            'macro_f1_mean',
            'min',
        ),
        macro_f1_across_algorithms_max=(
            'macro_f1_mean',
            'max',
        ),
        log_loss_across_algorithms_mean=(
            'log_loss_mean',
            'mean',
        ),
    )
    .reset_index()
)

representation_summary.to_csv(
    TABLES_DIR / 'Table_training_cv_representation_summary.csv',
    index=False,
    encoding='utf-8-sig',
)

display(representation_summary)


## 11. Figuras exploratorias de validación cruzada

In [ ]:

for dataset_id in DATASETS:
    plot_data = selected_configurations[
        selected_configurations['dataset_id'] == dataset_id
    ].copy()

    figure, axis = plt.subplots(figsize=(11, 6))

    algorithms = ALGORITHMS
    x = np.arange(len(REPRESENTATIONS))
    width = 0.18

    for algorithm_index, algorithm in enumerate(algorithms):
        part = (
            plot_data[
                plot_data['algorithm'] == algorithm
            ]
            .set_index('representation')
            .reindex(REPRESENTATIONS)
        )

        axis.bar(
            x + (
                algorithm_index
                - (len(algorithms) - 1) / 2
            ) * width,
            part['macro_f1_mean'],
            width=width,
            label=algorithm,
        )

    axis.set_xticks(x)
    axis.set_xticklabels(
        REPRESENTATIONS,
        rotation=30,
        ha='right',
    )
    axis.set_ylabel('Macro-F1 medio en CV de entrenamiento')
    axis.set_xlabel('Representación morfométrica')
    axis.set_title(
        f'Selección interna restringida a entrenamiento — {dataset_id}'
    )
    axis.legend(fontsize=8)
    axis.grid(True, axis='y', alpha=0.3)
    figure.tight_layout()

    figure.savefig(
        FIGURES_DIR / f'training_cv_macro_f1_{dataset_id}.png',
        dpi=300,
        bbox_inches='tight',
    )
    figure.savefig(
        FIGURES_DIR / f'training_cv_macro_f1_{dataset_id}.pdf',
        bbox_inches='tight',
    )
    plt.show()


## 12. Auditoría explícita de no utilización de calibración ni prueba

In [ ]:

partition_audit_rows = []

for dataset_id in DATASETS:
    reference = datasets[dataset_id][
        'R2_measured_core'
    ]

    for partition_name in [
        'train',
        'calibration',
        'test',
    ]:
        partition_audit_rows.append({
            'dataset_id': dataset_id,
            'partition': partition_name,
            'n_records': int(
                (
                    reference['partition']
                    == partition_name
                ).sum()
            ),
            'used_for_hyperparameter_selection': (
                partition_name == 'train'
            ),
        })

partition_use_audit = pd.DataFrame(
    partition_audit_rows
)

partition_use_audit.to_csv(
    RESULTS_DIR / '04_partition_use_audit.csv',
    index=False,
    encoding='utf-8-sig',
)

if partition_use_audit.loc[
    partition_use_audit['partition'].isin(
        ['calibration', 'test']
    ),
    'used_for_hyperparameter_selection',
].any():
    raise RuntimeError(
        'Calibración o prueba fueron marcados como usados.'
    )

display(partition_use_audit)


## 13. Resumen de tiempo de cómputo

In [ ]:

runtime_summary = (
    candidate_summary.groupby(
        ['dataset_id', 'representation', 'algorithm']
    )
    .agg(
        candidates=('candidate_id', 'nunique'),
        total_seconds=('total_seconds', 'sum'),
        mean_seconds_per_candidate=(
            'total_seconds',
            'mean',
        ),
    )
    .reset_index()
)

runtime_summary['total_minutes'] = (
    runtime_summary['total_seconds'] / 60.0
)

runtime_summary.to_csv(
    TABLES_DIR / 'Table_model_selection_runtime.csv',
    index=False,
    encoding='utf-8-sig',
)

display(
    runtime_summary.sort_values(
        'total_seconds',
        ascending=False,
    )
)


## 14. Validación final

In [ ]:

expected_selection_rows = (
    len(DATASETS)
    * len(REPRESENTATIONS)
    * len(ALGORITHMS)
)

final_checks = {
    'input_notebook03_validated': True,
    'only_training_used_for_selection': bool(
        (
            partition_use_audit.loc[
                partition_use_audit['partition'].isin(
                    ['calibration', 'test']
                ),
                'used_for_hyperparameter_selection',
            ] == False
        ).all()
    ),
    'all_candidates_have_three_folds': bool(
        (candidate_summary['folds'] == 3).all()
    ),
    'one_configuration_selected_per_cell': (
        len(selected_configurations)
        == expected_selection_rows
    ),
    'all_four_algorithms_selected': bool(
        (
            selected_configurations.groupby(
                ['dataset_id', 'representation']
            )['algorithm'].nunique() == 4
        ).all()
    ),
    'no_missing_selected_metrics': bool(
        selected_configurations[
            [
                'macro_f1_mean',
                'macro_f1_sd',
                'macro_f1_min',
                'log_loss_mean',
            ]
        ].notna().all(axis=None)
    ),
}

final_validation = pd.DataFrame([
    {
        'check': check,
        'passed': passed,
    }
    for check, passed in final_checks.items()
])

final_validation.to_csv(
    RESULTS_DIR / '05_final_model_selection_validation.csv',
    index=False,
    encoding='utf-8-sig',
)

if not final_validation['passed'].all():
    raise RuntimeError(
        'Falló al menos una validación final.'
    )

print('Validación final de selección: CORRECTA')
display(final_validation)


## 15. Crear ZIP y copiar a Google Drive

In [ ]:

zip_local = (
    LOCAL_ROOT
    / 'NOTEBOOK04_MODEL_SELECTION_OUTPUTS.zip'
)

if zip_local.exists():
    zip_local.unlink()

with zipfile.ZipFile(
    zip_local,
    'w',
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in OUTPUT_DIR.rglob('*'):
        if path.is_file():
            archive.write(
                path,
                arcname=path.relative_to(OUTPUT_DIR),
            )

zip_drive = (
    DRIVE_PROJECT_ROOT / '07_results'
    / 'NOTEBOOK04_MODEL_SELECTION_OUTPUTS.zip'
)

copied = copy_with_retry(
    zip_local,
    zip_drive,
    attempts=8,
)

print('Notebook 04 completado correctamente.')
print('ZIP local:', zip_local)
print('ZIP en Drive:', zip_drive)
print('Copia en Drive verificada:', copied)

if not copied:
    print(
        'Descarga manualmente el ZIP local desde el panel Archivos de Colab.'
    )



## Archivo que debes compartir

Al finalizar, envía:

```text
/content/drive/MyDrive/Grain_Robustness_Q1/07_results/
NOTEBOOK04_MODEL_SELECTION_OUTPUTS.zip
```

Si Drive falla:

```text
/content/Grain_Robustness_Q1_notebook04/
NOTEBOOK04_MODEL_SELECTION_OUTPUTS.zip
```

El notebook debe mostrar:

```text
Salida del Notebook 03 validada correctamente.
Evaluación de candidatos terminada.
Validación final de selección: CORRECTA
Notebook 04 completado correctamente.
```
